# 정차 계획 — 언제, 어느 휴게소에서 쉴 것인가

## 문제 정의

부산 → 인천공항은 약 423 km / 6시간입니다. 이 운행에는 **성격이 다른 두 개의 정차 요구**가 있습니다.

1. **법정 휴게** — 「화물자동차 운수사업법」상 **2시간 연속운전 시 15분 이상 휴식** (2021.3.1 시행,
   그 전에는 4시간/30분). 6시간 운행이면 최소 2회는 무조건 서야 합니다.
2. **충전** — 배터리가 하한(20%) 아래로 떨어지기 전에 충전해야 합니다.

**핵심은 이 둘을 같은 휴게소에서 겹치는 것입니다.** 충전은 어차피 차를 세워야 하고,
법정 휴게도 어차피 서야 합니다. 따로 서면 시간이 두 번 들고, 겹치면 한 번으로 끝납니다.

이 노트북은 각 휴게소의 **도착 시각·누적 연속운전시간·SOC·충전기 유무**를 함께 보고
정차 지점을 고릅니다. 출력은 "어디서 몇 분 쉬고, 그 중 충전이 몇 분인지"의 타임라인입니다.

> 법령은 개정될 수 있습니다. 실제 운행에 쓰기 전에 국가법령정보센터에서 현행 조문을 확인하세요.

## 0. 제원 · 법정 기준 · 운행 조건

In [5]:
import numpy as np
import pandas as pd

# ── Volvo FH Electric 공개 제원 (volvotrucks.com) ──────────────────
BATT_USABLE_KWH = 460.0    # "Up to 460 kWh"
OEM_RANGE_KM = 470.0       # "Up to 470 km"
CHG_MAX_KW = 350.0         # CCS 피크
CHG_EFF = 0.73             # 20→80% 65분에서 역산한 평균/피크 비

# ── 법정 휴게 기준 (화물자동차 운수사업법) ────────────────────────
MAX_DRIVE_MIN = 120.0      # 연속 운전 상한 (2021.3.1~ 2시간)
MIN_BREAK_MIN = 15.0       # 최소 휴게시간
# 구법 기준으로 보려면: MAX_DRIVE_MIN, MIN_BREAK_MIN = 240.0, 30.0

# ── 운행 조건 ──────────────────────────────────────────────────
GCW_KG = 40_000            # 총조합중량. 한국 도로법 상한 40t
SOC_START = 100.0
SOC_MIN = 20.0             # 안전 하한
SOC_CHARGE_TO = 80.0       # 급속충전 종료 SOC
AMBIENT_C = 5.0
DRIVING_STYLE = 0.35       # 0~1. 물류 정속주행이면 낮게
DEPART = pd.Timestamp("2026-01-15 06:00")   # 출발 시각

# ── 물리 상수 ──────────────────────────────────────────────────
G, RHO_AIR = 9.81, 1.2
CRR, CDA = 0.006, 5.5      # 구름저항계수 / Cd×A (트랙터+트레일러)
ETA_DRIVE, ETA_REGEN = 0.85, 0.60
AUX_KW = 3.0

oem_kwh100 = BATT_USABLE_KWH / OEM_RANGE_KM * 100
print(f"공인 전비 역산  {BATT_USABLE_KWH:.0f} kWh / {OEM_RANGE_KM:.0f} km = {oem_kwh100:.1f} kWh/100km")
print(f"법정 휴게       {MAX_DRIVE_MIN:.0f}분 연속운전마다 {MIN_BREAK_MIN:.0f}분 이상")
print(f"충전            20→80%({BATT_USABLE_KWH * 0.6:.0f} kWh) @ {CHG_MAX_KW:.0f} kW "
      f"= {BATT_USABLE_KWH * 0.6 / (CHG_MAX_KW * CHG_EFF) * 60:.0f}분")

공인 전비 역산  460 kWh / 470 km = 97.9 kWh/100km
법정 휴게       120분 연속운전마다 15분 이상
충전            20→80%(276 kWh) @ 350 kW = 65분


## 1. 물리 기반 전비 모델

승용차 합성 데이터(`ev_energy_consumption.csv`)는 전비가 평균 24 kWh/100km 로 이 트럭(공인 98)의
**4분의 1** 수준이고 적재량도 최대 500 kg 입니다. 절대값을 그대로 쓸 수 없어, 물리 모델이 절대 수준을 잡고
ML은 기온·운전스타일의 **상대 배율**만 담당하게 했습니다.

구배는 '구간 평균 %'가 아니라 **상승 누적 / 하강 누적**으로 받습니다. 회생 회수율이 60%라
오르막과 내리막이 상쇄되지 않기 때문입니다.

In [6]:
def segment_energy_kwh(dist_km, v_kmh, ascent_m=0.0, descent_m=0.0,
                       mass_kg=GCW_KG, aux_kw=AUX_KW):
    """구간 총 소모 에너지(kWh)."""
    v, d = v_kmh / 3.6, dist_km * 1000
    f_roll = CRR * mass_kg * G
    f_aero = 0.5 * RHO_AIR * CDA * v ** 2
    e_resist = (f_roll + f_aero) * d / ETA_DRIVE
    e_climb = mass_kg * G * ascent_m / ETA_DRIVE      # 오르막: 손실 포함 전량 소모
    e_regen = mass_kg * G * descent_m * ETA_REGEN     # 내리막: 회수율만큼만 회수
    e_aux = aux_kw * 1000 * (dist_km / v_kmh * 3600)
    return max(e_resist + e_climb - e_regen + e_aux, 0.0) / 3.6e6


def kwh_per_100km(v_kmh, mass_kg=GCW_KG, **kw):
    return segment_energy_kwh(100.0, v_kmh, mass_kg=mass_kg, **kw)


grid = pd.DataFrame({v: [round(kwh_per_100km(v, m * 1000), 1) for m in [25, 30, 40, 65]]
                     for v in [80, 90, 100]}, index=[f"{m}t" for m in [25, 30, 40, 65]])
grid.columns.name, grid.index.name = "속도 km/h", "총중량"
print(f"평지 정속 전비 (kWh/100km)   ※ 공인치 {oem_kwh100:.0f}\n{grid.to_string()}")
print(f"\n→ 공인치는 25~30t·80km/h 조건. 40t 만재 90km/h면 {kwh_per_100km(90, 40000):.0f} kWh/100km")

평지 정속 전비 (kWh/100km)   ※ 공인치 98
속도 km/h    80     90     100
총중량                         
25t      105.1  118.8  134.3
30t      114.7  128.4  143.9
40t      133.9  147.7  163.2
65t      182.0  195.8  211.2

→ 공인치는 25~30t·80km/h 조건. 40t 만재 90km/h면 148 kWh/100km


## 2. ML 보정계수 (기온 · 운전스타일)

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge

EV_CSV = "data/ev_energy_consumption.csv"
TARGET = "energy_consumption_kwhper100km"

ev = pd.read_csv(EV_CSV, encoding="utf-8-sig")
FEATURES = [c for c in ev.columns if c != TARGET]
ml = make_pipeline(StandardScaler(), PolynomialFeatures(2, include_bias=False),
                   Ridge(alpha=1.0)).fit(ev[FEATURES], ev[TARGET])

REF = {c: float(ev[c].median()) for c in FEATURES}
REF_PRED = float(ml.predict(pd.DataFrame([REF]))[0])


def hvac_from_temp(t):
    """쾌적구간(21°C) 에서 최소, 양쪽으로 증가."""
    return float(np.clip(abs(t - 21) * 0.18, 0, 5))


def ml_correction(speed_kmh, dist_km, temp_c=AMBIENT_C, style=DRIVING_STYLE):
    """기준 조건 대비 배율. 물리 모델이 이미 다루는 구배·적재는 기준값 고정."""
    row = dict(REF)
    row.update({
        "speed_kmh": float(np.clip(speed_kmh, ev["speed_kmh"].min(), ev["speed_kmh"].max())),
        "trip_distance_km": float(np.clip(dist_km, ev["trip_distance_km"].min(),
                                          ev["trip_distance_km"].max())),
        "ambient_temp_C": float(np.clip(temp_c, ev["ambient_temp_C"].min(),
                                        ev["ambient_temp_C"].max())),
        "hvac_power_kw": hvac_from_temp(temp_c),
        "driving_style_index": float(style),
    })
    return float(ml.predict(pd.DataFrame([row]))[0]) / REF_PRED


print(f"ML 학습 R2 = {ml.score(ev[FEATURES], ev[TARGET]):.4f} / 기준 예측 {REF_PRED:.2f}")
print("기온별 보정계수:", {f"{t}°C": round(ml_correction(90, 80, t), 3)
                            for t in [-10, 0, 10, 21, 30]})

ML 학습 R2 = 0.9529 / 기준 예측 23.44
기온별 보정계수: {'-10°C': 1.232, '0°C': 1.112, '10°C': 0.998, '21°C': 0.903, '30°C': 0.973}


## 3. 구간별 소모 · 주행시간

In [8]:
seg = pd.read_csv("busan_incheon_route_segments.csv", encoding="utf-8-sig")

for col in ["상승_m", "하강_m"]:
    if col not in seg.columns:
        seg[col] = 0.0
if (seg["상승_m"] == 0).all():
    print("경고: 고도 컬럼이 없어 평지로 처리합니다. 산악 구간은 과소평가됩니다.\n")

if "구간시간_분" not in seg.columns:
    seg["구간시간_분"] = seg["구간거리_km"] / seg["평균속도_kmh"] * 60


def predict_segment(row, mass_kg=GCW_KG, temp_c=AMBIENT_C, style=DRIVING_STYLE):
    base = segment_energy_kwh(row["구간거리_km"], row["평균속도_kmh"],
                              row["상승_m"], row["하강_m"], mass_kg)
    return base * ml_correction(row["평균속도_kmh"], row["구간거리_km"], temp_c, style)


seg["소모_kWh"] = [round(predict_segment(r), 1) for _, r in seg.iterrows()]
seg["전비_kWh100km"] = (seg["소모_kWh"] / seg["구간거리_km"] * 100).round(1)

drive_min = seg["구간시간_분"].sum()
print(f"총 주행거리 {seg['구간거리_km'].sum():.1f} km / 순수 주행시간 {drive_min / 60:.1f}시간")
print(f"총 소모 {seg['소모_kWh'].sum():.0f} kWh (가용 {BATT_USABLE_KWH:.0f} kWh 의 "
      f"{seg['소모_kWh'].sum() / BATT_USABLE_KWH * 100:.0f}%)")
print(f"법정 최소 휴게 횟수: {int(np.ceil(drive_min / MAX_DRIVE_MIN)) - 1}회 "
      f"(연속 {MAX_DRIVE_MIN:.0f}분 상한)")

# 한 구간이 연속운전 상한을 넘으면 구간 중간에 세울 곳이 필요합니다
over = seg[seg["구간시간_분"] > MAX_DRIVE_MIN]
if len(over):
    print(f"\n주의: {len(over)}개 구간이 단독으로 {MAX_DRIVE_MIN:.0f}분을 초과합니다.")
    print(over[["출발", "도착", "구간시간_분"]].to_string(index=False))
    print("→ 휴게소 탐색 간격(REST_SEARCH_STEP_KM)을 줄여 중간 정차 후보를 늘리세요.")

seg[["출발", "도착", "구간거리_km", "구간시간_분", "평균속도_kmh", "전비_kWh100km", "소모_kWh"]]

경고: 고도 컬럼이 없어 평지로 처리합니다. 산악 구간은 과소평가됩니다.



ValueError: Input X contains NaN.
PolynomialFeatures does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 4. 정차 계획

각 휴게소 도착 시점에 두 가지를 함께 봅니다.

- **다음 구간을 더 달리면 연속운전 상한을 넘는가** → 넘으면 여기서 쉬어야 합니다.
- **다음 충전 가능 휴게소까지 SOC가 버티는가** → 못 버티면 여기서 충전해야 합니다.

정차 시간은 **둘 중 긴 쪽**입니다. 충전이 40분 걸리면 법정 15분은 그 안에 흡수됩니다.
충전할 필요가 없으면 15분만 쉬고 갑니다.

충전이 필요한데 충전기가 없는 휴게소라면, 법정 휴게만 하고 다음 충전소까지 갈 수 있는지 다시 봅니다.
그것도 안 되면 완주 불가로 표시합니다.

In [ ]:
try:
    rest = pd.read_csv("busan_incheon_rest_areas_ev.csv", encoding="utf-8-sig")
    col = "충전가능" if "충전가능" in rest.columns else "충전기수"
    can_charge = dict(zip(rest["name"], rest[col] > 0))
    print(f"휴게소 {len(rest)}개 중 충전 가능 {sum(can_charge.values())}개 (판정 컬럼: {col})")
except FileNotFoundError:
    can_charge = {}
    print("휴게소 파일이 없어 모든 지점에서 충전 가능하다고 가정합니다.")


def plan_stops(seg, can_charge=None, soc_start=SOC_START, soc_min=SOC_MIN,
               soc_to=SOC_CHARGE_TO, max_drive=MAX_DRIVE_MIN, min_break=MIN_BREAK_MIN,
               charger_kw=CHG_MAX_KW, depart=DEPART):
    """법정 휴게와 충전을 함께 만족하는 정차 계획."""
    can_charge = {} if can_charge is None else can_charge
    n = len(seg)
    e = seg["소모_kWh"].values
    t = seg["구간시간_분"].values
    has_charger = [bool(can_charge.get(seg["도착"].iat[i], True)) for i in range(n)]
    if n:
        has_charger[n - 1] = False          # 목적지에서 충전은 의미 없음

    soc, since_break, clock = soc_start, 0.0, depart
    rows, feasible = [], True

    for i in range(n):
        clock += pd.Timedelta(minutes=float(t[i]))
        since_break += t[i]
        soc -= e[i] / BATT_USABLE_KWH * 100

        row = {"지점": seg["도착"].iat[i],
               "도착시각": clock.strftime("%H:%M"),
               "누적km": round(seg["구간거리_km"].values[:i + 1].sum(), 1),
               "연속운전_분": round(since_break),
               "도착SOC_%": round(soc, 1),
               "충전기": "O" if has_charger[i] else "-",
               "정차_분": 0, "그중충전_분": 0, "사유": ""}

        if soc < soc_min:
            row["사유"] = "!! SOC 하한 미달"
            feasible = False
            rows.append(row)
            continue

        if i == n - 1:
            rows.append(row)
            break

        # (1) 법정 휴게가 필요한가 — 다음 구간까지 달리면 상한 초과?
        need_break = since_break + t[i + 1] > max_drive

        # (2) 충전이 필요한가 — 다음 충전 가능 지점(없으면 종점)까지 버티나?
        nxt = next((m for m in range(i + 1, n - 1) if has_charger[m]), None)
        span = e[i + 1: (nxt + 1 if nxt is not None else n)].sum()
        need_charge = soc - span / BATT_USABLE_KWH * 100 < soc_min

        chg_min = 0.0
        reasons = []
        if need_charge:
            if has_charger[i]:
                kwh = (soc_to - soc) / 100 * BATT_USABLE_KWH
                chg_min = kwh / (charger_kw * CHG_EFF) * 60
                reasons.append(f"충전 {soc:.0f}→{soc_to:.0f}% ({kwh:.0f} kWh)")
                soc = soc_to
            else:
                reasons.append("!! 충전 필요하나 충전기 없음")
                feasible = False
        if need_break:
            reasons.append("법정 휴게")

        stop_min = max(chg_min, min_break if need_break else 0.0)
        if stop_min > 0:
            clock += pd.Timedelta(minutes=float(stop_min))
            since_break = 0.0
            row["정차_분"] = round(stop_min)
            row["그중충전_분"] = round(chg_min)
        row["사유"] = " + ".join(reasons)
        rows.append(row)

    plan = pd.DataFrame(rows)
    total_min = (clock - depart).total_seconds() / 60
    return plan, soc, total_min, feasible


plan, soc_end, total_min, ok = plan_stops(seg, can_charge)
n_stop = int((plan["정차_분"] > 0).sum())
print(f"\n{'완주 가능' if ok else '완주 불가'} | 도착 SOC {soc_end:.1f}% | 정차 {n_stop}회")
print(f"총 소요 {total_min / 60:.1f}시간 (주행 {seg['구간시간_분'].sum() / 60:.1f}h "
      f"+ 정차 {plan['정차_분'].sum() / 60:.1f}h)")
plan

## 5. 겹쳤을 때의 이득

법정 휴게와 충전을 **따로** 하면 각각의 시간이 그대로 더해집니다. **같은 휴게소에서 겹치면**
충전하는 동안 휴게 요건이 채워지므로 짧은 쪽이 흡수됩니다. 얼마나 절약되는지 비교합니다.

In [ ]:
charge_min = plan["그중충전_분"].sum()
stop_min = plan["정차_분"].sum()
n_break_needed = int(np.ceil(seg["구간시간_분"].sum() / MAX_DRIVE_MIN)) - 1
separate = charge_min + n_break_needed * MIN_BREAK_MIN

print(f"충전에 필요한 시간        {charge_min:5.0f}분")
print(f"법정 휴게 최소 소요       {n_break_needed * MIN_BREAK_MIN:5.0f}분 ({n_break_needed}회 × {MIN_BREAK_MIN:.0f}분)")
print(f"─────────────────────────────────")
print(f"따로 정차하면            {separate:5.0f}분")
print(f"겹쳐서 정차하면          {stop_min:5.0f}분")
print(f"절약                     {separate - stop_min:5.0f}분")

In [ ]:
# 시나리오별 정차 계획
scen = []
for label, mass, temp in [("공차 25t / 봄가을 15°C", 25_000, 15),
                          ("30t / 겨울 0°C", 30_000, 0),
                          ("40t 만재 / 겨울 -5°C", 40_000, -5),
                          ("40t 만재 / 여름 33°C", 40_000, 33)]:
    s = seg.copy()
    s["소모_kWh"] = [predict_segment(r, mass, temp) for _, r in s.iterrows()]
    p, soc_e, tot, okk = plan_stops(s, can_charge)
    scen.append({"시나리오": label,
                 "평균전비": round(s["소모_kWh"].sum() / s["구간거리_km"].sum() * 100, 1),
                 "정차횟수": int((p["정차_분"] > 0).sum()),
                 "정차시간_분": int(p["정차_분"].sum()),
                 "그중충전_분": int(p["그중충전_분"].sum()),
                 "총소요_시간": round(tot / 60, 1),
                 "도착SOC": round(soc_e, 1),
                 "완주": "O" if okk else "X"})

pd.DataFrame(scen)

## 6. 현실 점검

**충전 출력이 계획을 뒤집습니다.** 위 계산은 350 kW를 전제합니다.
2022년 기준 전국 고속도로 휴게소 충전기 860기 중 **82%가 100 kW 이하**, 200 kW 이상은 18%뿐이었습니다.
100 kW 충전기라면 20→80%에 65분이 아니라 **약 3시간 15분**이 걸립니다.
그러면 "쉬는 김에 충전"이 아니라 "충전 때문에 반나절 대기"가 됩니다.

**대형차 진입이 더 큰 변수입니다.** 트랙터+트레일러는 전장 16.5 m 이상인데 휴게소 충전 구획은
승용차 기준으로 설계돼 있습니다. 1번 노트북의 `충전기수` 는 "반경 안에 충전기가 존재한다"는 뜻이지
**"대형 트럭이 진입·주차해서 쓸 수 있다"가 아닙니다.** 후보 휴게소는 개별 확인이 필요합니다.

In [ ]:
print("충전기 출력별 20→80% (276 kWh) 소요시간과 정차 계획 영향\n")
for kw, note in [(350, "제원상 65분"), (200, "휴게소 상위 18%"),
                 (100, "휴게소 82%가 이 이하"), (50, "구형")]:
    eff = CHG_EFF if kw == 350 else 0.85
    p, soc_e, tot, okk = plan_stops(seg, can_charge, charger_kw=kw)
    print(f"  {kw:3d} kW → 20~80% {276 / (kw * eff) * 60:5.0f}분 | "
          f"총소요 {tot / 60:4.1f}h | 정차 {int(p['정차_분'].sum()):4d}분  ({note})")

---

## 정리

**믿을 만한 것**
- 법정 휴게 제약 처리. 연속운전 시간을 구간마다 누적해 상한 전에 세웁니다.
- 물리 모델의 절대 전비 수준. 공인치를 25~30t 조건으로 재현합니다.
- 시나리오 간 상대 비교 (만재 대비 공차, 겨울 대비 여름).

**믿기 어려운 것**
- ML 보정계수. 승용차 합성 데이터의 상대 민감도를 트럭에 전이한 가정입니다.
- 평지 가정 하의 구간 전비. 고도를 붙이기 전까지 산악 구간은 과소평가됩니다.
- 각 휴게소의 실제 충전 가능 여부와 출력. 지금은 "충전기 존재" 수준입니다.

**다음 순서**
1. 1번 노트북 실행 → 실제 휴게소 위치와 구간표 확보
2. 고도(DEM) 연동 → `상승_m` / `하강_m` 컬럼 추가
3. 후보 휴게소의 충전기 출력·대형차 진입 가능 여부 개별 확인
4. 기상청 ASOS 연동 → 구간별 실제 기온